In [ ]:
import re
import string
import numpy as np
import pandas as pd
import csv
import json
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

In [ ]:
verbs = ["agree", "believe", "bet", "consider", "decide", "expect", "feel",
         "figure", "find", "guess", "hear", "hope", "imagine", "know", "mean",
         "notice", "read", "realize", "remember", "say", "see", "show",
         "suppose", "take", "teach", "tell", "thank", "think", "understand",
         "wish", "worry"]
additional_verbs = ["accept", "acknowledge", "add", "affirm", "allege",
                    "announce", "answer", "anticipate", "argue", "assert",
                    "attest", "boast", "brag", "calculate", "caution",
                    "certify", "claim", "comment", "complain", "conceal",
                    "concede", "confess", "confirm", "declare", "deduce",
                    "demonstrate", "deny", "determine", "deny", "determine",
                    "disclose", "doubt", "dream", "emphasize", "establish",
                    "estimate", "explain", "fear", "forget", "gloat", "growl",
                    "guarantee", "hate", "hint", "holler", "hoot", "hypothesize",
                    "ignore", "imply", "indicate","infer", "insist", "intimate",
                    "joke", "learn", "maintain", "mention", "moan",
                    "mumble", "murmur", "muse", "mutter", "note", "observe",
                    "opine", "perceive", "plead", "predict", "presume", "pretend",
                    "proclaim", "promise", "propose", "prove", "reason", "recall",
                    "reckon", "recognize", "recollect", "regret", "reiterate",
                    "remark", "repeat", "reply", "report", "request", "resent",
                    "respond", "reveal", "scream", "sense", "shout", "shriek",
                    "signal", "signify", "speculate", "stammer", "state",
                    "suggest", "suspect", "swear", "testify", "theorize", "trust",
                    "verify", "vow", "wail", "warn", "whine", "whisper", "wonder",
                    "write", "yell"]
all_verbs = verbs + additional_verbs
all_verbs_set = set(all_verbs)
len(all_verbs)

In [ ]:
# constituency parsing
!pip install spacy
!python -m spacy download en_core_web_md
# !python -m spacy download en_core_web_trf
# !pip install -U spacy[cuda12x]
# !python -m spacy download en_core_web_sm

In [ ]:
import spacy

# spacy.prefer_gpu()
spacy.require_cpu() # default to using cpu

nlp = spacy.load("en_core_web_md")
# nlp = spacy.load("en_core_web_trf")
# nlp = spacy.load("en_core_web_sm")


wh_words = ["which", "how", "what", "why", "when", "where", "who", "whom", "whose"]

def get_span(left_id, right_id, doc):
  return doc[left_id:right_id+1]

def get_subj_type(subj):
  if subj.lemma_.lower() == "i":
    return "I"
  elif subj.lemma_.lower() == "you":
    return "You"
  elif subj.pos_ == "PRON":
    return "pronoun"
  else:
    return "NP"

# check control/raising
def check_xcomp(verb):
  return verb.dep_ == "xcomp" or any(a.dep_ == "xcomp" for a in verb.ancestors)

# get the subject of the verb
def get_overt_subject(verb):
  for child in verb.children:
    if child.dep_.startswith("nsubj") or child.dep_.startswith("csubj") or child.dep_ == "expl":
      return child
  return None

def extract_subject(verb):
  """
  Get the subject of the verb by:
    1) checking overt subject on verb
    2) going through cordination to find a governing verb/AUX that has a subject.
    ignore cases like raising/control: VERB <-xcomp- VERB, VERB <-aux/-auxpass- AUX
  Return: subject
  """
  # 1) finding the overt subject on the verb
  subject = get_overt_subject(verb)
  if subject:
    return subject

  # 2) finding the governing verb/AUX with a subject
  #    Typical chains: VERB <-xcomp- VERB, VERB <-aux/-auxpass- AUX
  current_word = verb
  visited = set()
  while current_word.head is not current_word and current_word.i not in visited:
    visited.add(current_word.i)
    # only find if the verb is coordinated with another verb
    if current_word.dep_ == "conj" and current_word.head.pos_ in {"VERB","AUX"}:
      parent = current_word.head
      if parent.pos_ in {"VERB","AUX"}:
        subject = get_overt_subject(parent)
        if subject:
          return subject
          current_word = parent
          continue
    break
  return None

# get the object of the verb
def extract_object(verb):
  for child in verb.children:
    if "obj" in child.dep_ or "dative" in child.dep_:
      return child
  return None

# get the modification of the verb
def extract_mod(verb):
  for child in verb.children:
    if "mod" in child.dep_:
      return child
  return None

def get_embedded_clause(embedded_verb):
  """
  Check the type of complementizer and get the span (left edge id and right edge id) of the embedded clause
  Return:
    comp_type: "that" for overt 'that' complementizer, "omitted" for omitted complementizer, "other" for other complementizer types
    left_i: the left edge of the embedded clause (including the complementizer if there is one), None if it's other complementizer
    right_i: the right edge of the embedded clause (including the complementizer if there is one), None if it's other complementizer
  """
  left_i = embedded_verb.left_edge.i
  right_i = embedded_verb.right_edge.i

  that_complementizer = None
  other_complementizer = None

  for ch in embedded_verb.children:
    # exclude other types of complementizer
    if ch.dep_ == "mark":
      if ch.lemma_.lower() == "that":
        if that_complementizer is None or ch.i < that_complementizer.i:
          that_complementizer = ch
      else:
        other_complementizer = ch
  # the subtree of the embedded verb does not include the complementizer
  # so the leftmost boundary should be the complementizer
  if that_complementizer:
    left_i = min(that_complementizer.i, left_i)
    return "that", left_i, right_i
  # if other complementizer is used
  if other_complementizer:
    return "other", None, None
  # if the complementizer is omitted
  return "omitted", left_i, right_i


def extract_ccomps(doc):
  """
  return one record per (matrix verb, ccomp head) where the complementizer is 'that' or omitted.
  Ignores xcomp entirely (not returned). Still robust to coordination and non-root predicates.
  """
  rows = []
  for sent in doc.sents:
    for tok in sent:
      if tok.pos_ in {"VERB","AUX"} and tok.lemma_.lower() in all_verbs_set: # tok is the matrix predicate
        embedded_clause = False
        comp_type = "none" # default to no embedded clause

        # extract the subject
        matrix_subj = extract_subject(tok)
        matrix_subj_left_i = matrix_subj.left_edge.i if matrix_subj else None
        matrix_subj_right_i = matrix_subj.right_edge.i if matrix_subj else None
        matrix_subj_span = get_span(matrix_subj_left_i, matrix_subj_right_i, matrix_subj.doc) if matrix_subj else None
        matrix_span_no_verb = get_span(tok.left_edge.i, tok.i-1, tok.doc) if tok.i != tok.left_edge.i else get_span(0, tok.i-1, tok.doc)
        matrix_span_current_no_verb = get_span(matrix_subj_left_i, tok.i-1, tok.doc) if matrix_subj else get_span(0, tok.i-1, tok.doc)

        # in case of control/raising: skip for now
        if check_xcomp(tok):
          continue
        
        matrix_span_verb = str(matrix_span_no_verb) + " " + tok.text
        matrix_span_current_verb = str(matrix_span_current_no_verb) + " " + tok.text

        # check if the verb has a ccomp
        for ch in tok.children:
          # the child is the embedded verb, connected to the matrix verb by ccomp (i.e., matrix verb -ccomp-> embedded verb)

          if ch.dep_ == "ccomp":
            comp_type, embedded_clause_left_i, embedded_clause_right_i = get_embedded_clause(ch)
            if comp_type == "other":
              continue  # skip other complementizers for now

            embedded_clause = True
            embedded_subj = extract_subject(ch)
            
            embedded_clause_span = get_span(embedded_clause_left_i, embedded_clause_right_i, ch.doc)
            # if the embedded clause starts with wh-words, i.e., taking a wh-constituent, skip for now
            # simple way: just check if the wh word is the first word of the embedded clause
            if str(embedded_clause_span).split()[0].lower() in wh_words :
              comp_type = "other"
              continue
            # more complicated: check if the wh-word is the object/modifier of the verb
            if str(extract_object(ch)) in wh_words or str(extract_mod(ch)) in wh_words:
              comp_type = "other"
              continue
            
            embedded_clause_omit_that_left_i = embedded_clause_left_i + 1 if comp_type == "that" else embedded_clause_left_i # left edge of the comp, not including "that" if there is one

            # onset of the embedded clause use the right bound of embedded subj if there is an embedded subj, else use the position of the embedded verb (not including the verb, henche the ch.i-1)
            cc_onset_span = get_span(embedded_clause_omit_that_left_i, embedded_subj.right_edge.i, ch.doc) if embedded_subj else get_span(embedded_clause_omit_that_left_i, ch.i-1, ch.doc)
            # get first word of embedded clause, no quotation mark even if it is a quote
            embedded_clause_one_word = ch.doc[embedded_clause_omit_that_left_i] if embedded_clause_right_i - embedded_clause_left_i  >= 1 else embedded_clause_span
            
            matrix_predicate_to_cc = embedded_clause_left_i - tok.i - 1

            # remove direct quotations
            word_after_verb = str(get_span(tok.i+1, tok.i+matrix_predicate_to_cc, tok.doc)) # if it is a quote then this contains a quotation mark
            comp_type = "quote" if '"' in word_after_verb or "'" in word_after_verb else comp_type
            
            rows.append({
                # sentence
                "sentence": sent.text,

                # matrix predicate
                "matrix_predicate": tok.text,
                "matrix_predicate_lemma": tok.lemma_,
                "matrix_predicate_id": tok.i, # absolute position of the verb in the sent
                "matrix_predicate_position": tok.i - tok.left_edge.i, # relative position of the verb in the matrix sentence

                # matrix subject
                "matrix_subject_head": matrix_subj.text if matrix_subj else None,
                "matrix_subject_head_id": matrix_subj.i if matrix_subj else None,
                "matrix_subject_span": str(matrix_subj_span) if matrix_subj else None,
                "matrix_subject_type": get_subj_type(matrix_subj) if matrix_subj else None,

                # matrix clause
                "matrix_span_no_verb": str(matrix_span_no_verb),
                "matrix_span_verb": matrix_span_verb,
                "matrix_span_current_no_verb": str(matrix_span_current_no_verb),
                "matrix_span_current_verb": matrix_span_current_verb,

                # complementizer
                "complementizer": comp_type,
                "complement_type": "ccomp",
                "matrix_predicate_to_cc":matrix_predicate_to_cc,

                # embedded clause
                # use the right bound of embedded subj if there is an embedded subj, else use the position of the embedded verb, should be the same as len(cc_onset_span)
                "cc_onset": embedded_subj.right_edge.i - embedded_clause_omit_that_left_i + 1 if embedded_subj else ch.i - embedded_clause_omit_that_left_i,
                "cc_onset_span": str(cc_onset_span) if cc_onset_span else None,
                "cc_remainder": embedded_clause_right_i - embedded_subj.right_edge.i if embedded_subj else embedded_clause_right_i - ch.i + 1, # need to include the embedded verb
                "embedded_clause_minus_one": embedded_clause_right_i - embedded_clause_omit_that_left_i,
                "embedded_clause_span": str(embedded_clause_span) if embedded_clause_span else None,
                "embedded_subject_head": embedded_subj.text if embedded_subj else None,
                "embedded_subject_head_id": embedded_subj.i if embedded_subj else None,
                "embedded_subject_span": str(get_span(embedded_subj.left_edge.i, embedded_subj.right_edge.i, embedded_subj.doc)) if embedded_subj else None,

                "one_word_omit_that": str(matrix_span_verb) + " " + str(embedded_clause_one_word) if matrix_predicate_to_cc == 0 or comp_type == "quote" else str(matrix_span_verb) + " " + word_after_verb + " " + str(embedded_clause_one_word),
                "one_word_with_that": str(matrix_span_verb) + " that " + str(embedded_clause_one_word) if matrix_predicate_to_cc == 0 or comp_type == "quote" else str(matrix_span_verb) + " " + word_after_verb + " that " + str(embedded_clause_one_word),
                "one_word_current_omit_that" : str(matrix_span_current_verb) + " " + str(embedded_clause_one_word) if matrix_predicate_to_cc == 0 or comp_type == "quote" else str(matrix_span_current_verb) + " " + word_after_verb + " " + str(embedded_clause_one_word),
                "one_word_current_with_that" : str(matrix_span_current_verb) + " that " + str(embedded_clause_one_word) if matrix_predicate_to_cc == 0 or comp_type == "quote" else str(matrix_span_current_verb) + " " + word_after_verb + " that " + str(embedded_clause_one_word)

                })

        # if there is no child with ccomp (i.e., no embedded clause) or if it is "whether" or other complementizer
        if not embedded_clause:
          rows.append({
              # sentence
              "sentence": sent.text,

              # matrix predicate
              "matrix_predicate": tok.text,
              "matrix_predicate_lemma": tok.lemma_,
              "matrix_predicate_id": tok.i, # absolute position of the verb in the sent
              "matrix_predicate_position": tok.i - matrix_subj_right_i if matrix_subj else None, # relative position of the verb in the matrix sentence (i.e. # of words after the matrix subject)

              # matrix subject
              "matrix_subject_head": matrix_subj.text if matrix_subj else None,
              "matrix_subject_head_id": matrix_subj.i if matrix_subj else None,
              "matrix_subject_span": str(matrix_subj_span) if matrix_subj else None,
              "matrix_subject_type": get_subj_type(matrix_subj) if matrix_subj else None,

              # matrix clause
              "matrix_span_no_verb": str(matrix_span_no_verb),
              "matrix_span_verb": matrix_span_verb,
              "matrix_span_current_no_verb": str(matrix_span_current_no_verb),
              "matrix_span_current_verb": matrix_span_current_verb,

              # complementizer and embedded clause
              "complementizer": comp_type,
              "complement_type": None,
              "matrix_predicate_to_cc": None,

              "cc_onset": None,
              "cc_onset_span": None,
              "cc_remainder": None,
              "embedded_clause_minus_one": None,
              "embedded_clause_span": None,
              "embedded_subject_head": None,
              "embedded_subject_head_id": None,
              "embedded_subject_span": None,

              "one_word_omit_that": None,
              "one_word_with_that": None,
              "one_word_current_omit_that": None,
              "one_word_current_with_that": None
          })
  return rows

In [ ]:
# downgrade to 3.6.0 to run dolma dataset
# see: https://github.com/huggingface/datasets/issues/7693
# !pip install datasets
!pip install datasets==3.6.0

In [ ]:
from datasets import load_dataset
# 13,095,416 sentences in total
ds = load_dataset("allenai/dolma","v1_6-sample",split="train")

In [ ]:
# remove sentences from The Stack (source=="stack-dedup") since most are code-related
# this results in 12,462,749 sentences (originally: 13,095,416)
def filter_out_source(batch, source):
  return [t != source for t in batch["source"]]

ds_filter = ds.filter(lambda batch: filter_out_source(batch, "stack-dedup"), batched=True, batch_size=10000)

In [ ]:
ds_filter_shuffle = ds_filter.shuffle(seed=1024)
ds_filter_shuffle = ds_filter_shuffle.add_column("doc_id", list(range(len(ds_filter_shuffle))))

In [ ]:
# ds_filter_shuffle.to_csv("/content/drive/MyDrive/comp_drop/dolma_v1_6-sample.csv")
ds_sentences_1 = ds_filter_shuffle.select(range(500000))
ds_sentences_2 = ds_filter_shuffle.select(range(500000,1000000))
ds_sentences_3 = ds_filter_shuffle.select(range(1000000,1500000))
ds_sentences_4 = ds_filter_shuffle.select(range(1500000,2000000))
# ds_sentences_1 = ds_shuffle.take(1000000) # if using streaming when loading the dataset

In [ ]:
from datasets import load_dataset
ds_sentences_1 = load_dataset("csv", data_files="/content/drive/MyDrive/comp_drop/dolma_v1_6-sample_1.csv")["train"]

In [ ]:
_split_re = re.compile(r'(?:\n|\s{2,}|(?<=[.!?])\s+)')

def clean_sent(sent):
  # replace special symbols
  sent = sent.replace("’", "'")
  sent = sent.replace("“", "\"")
  sent = sent.replace("”", "\"")
  sent = sent.replace("…", "...")
  sent = sent.replace("–", "-")
  # remove https like texts
  sent = re.sub(r'.*?(https?://[^\s]+)?\s*#?\w*\{[^}]*\}\s*', '', sent).strip()
  sent = re.sub(r'\[([^\]]+)\]\(https?://[^\)]+\)', r'\1', sent)
  sent = re.sub(r'https?://\S+', '', sent)
  # remove leading bullet-like patterns such as "*.", "-", etc.
  return re.sub(r'^\s*[\*\-]+\s*\.?\s*', '', sent).strip()

# if only need to clean but not split
def clean_batch(batch):
  texts = batch["sentence"]
  cleaned_texts = [clean_sent(t) for t in texts]
  return {"sentence": cleaned_texts}

# clean and split
def split_clean_sent(batch):
  texts = batch["text"]
  doc_ids = batch["doc_id"]
  sources = batch["source"]

  sentence = []
  doc_id = []
  sent_id = []
  sent_source = []

  for id, source, text in zip(doc_ids, sources, texts):
    if not text:
      continue
    sentences = _split_re.split(str(text))
    sentences = [clean_sent(sent) for sent in sentences if clean_sent(sent)]

    for i, sent in enumerate(sentences):
      sentence.append(sent)
      doc_id.append(id)
      sent_id.append(i)
      sent_source.append(source)

  return {"sent_id_in_doc": sent_id, "sentence": sentence, "doc_id": doc_id, "source": sent_source}

In [ ]:
# clean and split
# 5019354 sentences
ds_sentences_1_split = ds_sentences_1.map(
    split_clean_sent,
    batched=True,
    remove_columns=ds_sentences_1.column_names,
    batch_size=128,
    num_proc=4,
    )

In [ ]:
# only clean
ds_sentences_2_clean = ds_sentences_2.map(
    clean_batch,
    batched=True,
    remove_columns=["sentence"],
    batch_size=128,
    num_proc=4
)

In [ ]:
# 12,441,615 sentences in ds_sentences_1_split, load 1,000,000 to ds_sentences_test
# ds_sentences_1_split.to_csv("/content/drive/MyDrive/comp_drop/dolma_v1_6-sample_1_split.csv")
# from datasets import Dataset
# ds_sentences_test = Dataset.from_dict(ds_sentences_1_split[:1000000])

from datasets import load_dataset

ds_sentences_1_split = load_dataset("csv", data_files="/content/drive/MyDrive/comp_drop/dolma_v1_6-sample_1_split.csv")["train"]
ds_sentences_test = ds_sentences_1_split.select(range(1000000))

In [ ]:
def process_batch(batch):
  sentences = batch["sentence"]
  doc_ids = batch["doc_id"]
  sent_ids = batch["sent_id_in_doc"]
  sources = batch["source"]

  # process the sentences
  all_rows = []
  for doc, sent_id, doc_id, source in zip(nlp.pipe(sentences), sent_ids, doc_ids, sources):
    for sent in doc.sents:
      sent_doc = sent.as_doc() # the doc of each sentence to avoid index problems
      results = extract_ccomps(sent_doc)
      if not results:
        continue
      for r in results:
        if not isinstance(r, dict):
          raise TypeError(f"Expected dict, got {type(r)}")
        r.setdefault("sentence", sent.text.strip())
        r["doc_id"] = doc_id
        r["sent_id_in_doc"] = sent_id
        r["source"] = source
      all_rows.extend(results)

  if not all_rows:
    return {"sentence": [], "sent_id_in_doc":[], "doc_id":[], "source":[]}

  # add the keys
  all_keys = set()
  for r in all_rows:
    all_keys.update(r.keys())

  # put "sentence" as the first column and reorder the rest
  ordered_keys = (["sentence"] if "sentence" in all_keys else []) + sorted(k for k in all_keys if k != "sentence")
  result = {k: [r.get(k) for r in all_rows] for k in ordered_keys}

  return result

In [ ]:
# 1,000,000 sentences took 14min
if __name__ == "__main__":
  ds_processed_test = ds_sentences_test.map(
    process_batch,
    batched=True,
    remove_columns=ds_sentences_test.column_names,
    batch_size=128,
    num_proc=4,
    )

In [ ]:
ds_processed_test.to_csv("/content/drive/MyDrive/comp_drop/dolma_v1_6-sample_test_processed.csv")